In [4]:
import requests
import pandas as pd
from pathlib import Path

season = 2627
# Fetch bootstrap data
bootstrap_url = 'https://fantasy.premierleague.com/api/bootstrap-static/'
bootstrap_response = requests.get(bootstrap_url)
bootstrap_data = bootstrap_response.json()

# Extract current gameweek 

# Better way: find the CURRENT gameweek (the one that's finished or in progress)
current_gw = next(
    (event['id'] for event in bootstrap_data['events'] if event['is_current']),
    None
)

print(f"Current Gameweek: {current_gw}")


Current Gameweek: 3


In [5]:
 

# ============================================
# 1. FETCH DATA
# ============================================
print(f"Fetching FPL bootstrap data for season {season}...")
bootstrap_url = 'https://fantasy.premierleague.com/api/bootstrap-static/'
bootstrap_response = requests.get(bootstrap_url)
bootstrap_data = bootstrap_response.json()

# ============================================
# 2. GET CURRENT GW
# ============================================
current_gw = next(
    (event['id'] for event in bootstrap_data['events'] if event['is_current']),
    None
)

print(f"Processing GW {current_gw}, Season {season}")

# ============================================
# 3. EXTRACT PRICE CHANGES
# ============================================
elements = bootstrap_data['elements']
price_changes = []

for player in elements:
    price_changes.append({
        'season': season,  # Add season as numeric column
        'gameweek': current_gw,
        'player_code': player['code'],
        'web_name': player['web_name'],
        'now_cost': player['now_cost'] / 10,
        'cost_change_event': player['cost_change_event'],
        'cost_change_event_fall': player['cost_change_event_fall'],
        'price_change_status': 'up' if player['cost_change_event'] > 0 else ('down' if player['cost_change_event'] < 0 else 'stable'),
        'price_change_percent': player['price_change_percent']
    })

# ============================================
# 4. SAVE TO FOLDER STRUCTURE
# ============================================
df_price_changes = pd.DataFrame(price_changes)



Fetching FPL bootstrap data for season 2627...
Processing GW 3, Season 2627


In [6]:
df_price_changes

,season,gameweek,player_code,web_name,now_cost,cost_change_event,cost_change_event_fall,price_change_status,price_change_percent
0,2627,3,154561,Raya,6.0,0,0,stable,4.3
1,2627,3,109745,Arrizabalaga,4.9,-1,1,down,-0.7
2,2627,3,437495,Meslier,4.9,-1,1,down,-0.6
3,2627,3,226597,Gabriel,8.0,0,0,stable,-43.0
4,2627,3,445122,J.Timber,6.5,0,0,stable,-0.2
...,...,...,...,...,...,...,...,...,...
647,2627,3,437505,Isidor,5.5,0,0,stable,5.1
648,2627,3,638411,Methalie,4.5,0,0,stable,0.0
649,2627,3,696104,Ahoka,4.5,0,0,stable,0.0
650,2627,3,549940,Fofana,5.5,0,0,stable,0.0


In [8]:
# Look at what fields each player actually has
player = bootstrap_data['elements'][0]
print(player.keys())

dict_keys(['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'price_change_hourly_rate', 'price_change_projections', 'price_change_locked_until', 'price_change_calibrating', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 're

In [7]:
df_price_changes['price_change_status'].value_counts()

price_change_status
stable    628
down       23
up          1
Name: count, dtype: int64

In [ ]:
# Create filename: {season}_price_change_gw{gw}.csv
filename = f'{season}_price_change_gw{current_gw}.csv'

# Create data folder path
data_path = Path(r'C:\Users\JesseOnu\fpl sql rework\data')
data_path.mkdir(parents=True, exist_ok=True)

# Save file
output_file = data_path / filename
df_price_changes.to_csv(output_file, index=False)

print(f"✓ Saved {len(df_price_changes)} price changes to {output_file}")